# Combinatoria de experimentos, partiendo del deposito real

La grilla de `old_base/05_combinatoria` media cada plan aplicado de una vez al deposito. Esta
repite la idea con la simulacion que el proyecto usa desde `correct_base/01_transicion`: el deposito
arranca como esta al corte y el plan entra por las mudanzas. El objetivo es **descartar las opciones
que no suman y elegir el valor mas util de cada factor**: la frecuencia $f$, la distancia $D$, la
afinidad, el slotting, el batching y la regla de reposicion.

**Contenido.**
0. Configuracion.
1. El regimen de simulacion y las dos referencias.
2. Los factores: que es cada uno y que valores toma.
3. El diseno: la configuracion base, los bloques y la lista de tareas.
4. La corrida.
5. Controles.
6. Bloque A: slotting por ventana.
7. Bloque R: regla de reposicion.
8. Bloque F: frecuencia.
9. Bloque Q: parametros del QAP (`swap_search`).
10. Bloque B: algoritmo de batching.
11. Bloque G: recorrido.
12. Que queda y que se descarta.

## 0. Configuracion

In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from abs_affinity_based_slotting.config import PROJECT_ROOT
from abs_affinity_based_slotting.experiments import (EvalSpec, InstanceSpec, SlottingSpec, Task,
                                                     run_tasks, load_results, load_steps, register_results)

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 80)

OUT = PROJECT_ROOT / "results" / "correct_base_02_combinatoria"   # resultados (fuera de git)

## 1. El regimen de simulacion y las dos referencias

Cada evaluacion simula el periodo de test, el ultimo 40% de los batches (72 dias, unas 10 semanas), con las
lineas de picking y las reposiciones de los datos, y mide la caminata de cada batch con el recorrido
en S. Lo que se elige en cada evaluacion es:

- **Desde donde arranca** (`start`). `real`: el deposito como esta al corte. `plan`: el plan aplicado
  de una vez al corte, que es el **techo**.
- **A donde va cada reposicion** (`replenishment`). `plan_aisle` es la regla del proyecto: si al SKU le
  quedan unidades se repone en el lugar, y si se agoto va a su pasillo del plan. Sus variantes y las
  otras politicas estan en la seccion 2.
- **Como se arman los batches**: re-batching por pasillos con ventana $W$ ($W = 1$ es FIFO), armado
  pool por pool con el pasillo que cada SKU tiene en ese momento.

**Las dos referencias.**
- **Hoy**: el deposito real con reposicion `random`, como se generaron los datos, y batches FIFO. Todas
  las diferencias de este notebook son contra hoy, en el mismo deposito y con el mismo recorrido.
- **Techo**: el plan aplicado de una vez, con reposicion `home`; lo que el plan daria si el deposito se
  reordenara entero el primer dia.

**Dos medidas por evaluacion.** La caminata media por batch en todo el test (*promedio*) y la de los
batches de los ultimos 14 dias (*final*). Partiendo del deposito real, la primera incluye la
transicion y la segunda muestra hasta donde se llega.

## 2. Los factores

**Deposito.** Los cuatro de `data/v2`.

| deposito | pasillos | merchants | que lo distingue |
|---|---|---|---|
| `small` | 4 | 4 | el mas chico; un batch cada 240 minutos |
| `medium` | 8 | 50 | el principal |
| `medium-alpha-1.3` | 8 | 50 | `medium` con la demanda mas concentrada en pocos SKUs |
| `big` | 16 | 100 | el mas grande; un batch cada 14,4 minutos |

**Frecuencia $f_i$** (instancia): que se cuenta como demanda de un SKU en train.

| valor | que cuenta |
|---|---|
| `pick_lines` | lineas de picking |
| `unique_orders` | ordenes distintas que lo piden (igual a lineas: ninguna orden repite un SKU) |
| `unique_batches` | batches distintos que lo visitan |
| `total_units` | unidades |

**Afinidad $a_{ij}$** (instancia): cuanto aparecen juntos dos SKUs en train. La usan `swap_search` y la
reposicion `objective_greedy`.

| parametro | valores | que es |
|---|---|---|
| metrica | `jaccard`, `cosine`, `cooccurrence` | como se normaliza el conteo de apariciones juntas |
| unidad | `order_id`, `batch_id` | aparecer juntos en la misma orden o en el mismo batch |
| filtro | `top_k:5`, `top_k:10`, `top_k:20`, `mutual_top_k:10` | cuantos vecinos se conservan por SKU |
| normalizacion | si, no | si $f$ y $a$ se pasan a proporciones para que los dos terminos del QAP queden en la misma escala |

**Distancia entre productos $D$** (instancia), la que cobra el termino cuadratico: `aisle` (cero en el
mismo pasillo, un pasillo entero mas el cruce si no) o `walking` (caminando de bay a bay).

**Slotting** (plan).

| plan | que hace |
|---|---|
| vigente | cada SKU donde estaba al corte |
| `demand_greedy` | SKUs por $f_i$ descendente a ubicaciones por distancia al dock |
| `linear_assignment` | el optimo exacto de $\sum_i f_i\, c_{l(i)}$; no corre en `big` por memoria |
| `aisle_fill` | SKUs por $f_i$ descendente, llenando un pasillo entero antes del siguiente |
| `merchant_aisle_fill` | igual, con los SKUs agrupados por merchant (merchants por demanda total) |
| `bilevel` (grupo/zona) | agrupa los SKUs (por merchant o en clases A, B y C), un solver le da a cada grupo un bloque de ubicaciones (franjas por distancia al dock, `dock`, o pasillos consecutivos, `aisle_order`), y adentro aplica `demand_greedy` |
| `swap_search` | desde un plan inicial, intercambios y mudanzas que bajan $\lambda \sum_i f_i c_{l(i)} + (1 - \lambda)\sum_{i,j} a_{ij} D(l(i), l(j))$; $\lambda = 0$ es solo afinidad |

**Batching** (evaluacion): la ventana $W$ (1, 2, 4, 8 y 16) y las pasadas de intercambio del algoritmo
(0, solo construir; 20, construir e intercambiar).

**Recorrido** (evaluacion): `s_shape`, el de la operacion, o `snake`, que vuelve al frente por el mismo
pasillo y cobra cuanto se entra en cada uno.

**Reposicion** (evaluacion).

| politica | que hace |
|---|---|
| `plan_aisle` | la regla; `rule_trigger`: `stockout` (mueve solo si el SKU se agoto) o `always` (en cada reposicion, aunque le queden unidades); `rule_target`: `aisle` (a su pasillo del plan) o `location` (a su ubicacion exacta del plan, o la libre mas cercana) |
| `objective_greedy` | sin plan: a la ubicacion libre o propia que menos sube el objetivo del QAP con el $\lambda$ dado; con $\lambda = 0$ decide solo por afinidad |
| `random` | si quedan unidades en el lugar; si se agoto, con probabilidad 0,9 a una ubicacion libre al azar (hoy) |
| `home` | a la ubicacion del plan (solo con el plan aplicado de una vez: el techo) |

## 3. El diseno

Todas las combinaciones son cientos de miles de corridas. El diseno tiene una grilla completa con los
dos factores que mas mueven el resultado, el slotting y la ventana (bloque A), y barridos de a un
factor para el resto: se parte de la **configuracion base** y se cambia un solo factor por vez. Los
barridos muestran el efecto de cada factor con todo lo demas fijo; no muestran como interactuan dos
factores barridos.

**Configuracion base.** Frecuencia `pick_lines`; afinidad `jaccard` por orden con `top_k:10`,
normalizada; $D$ = `aisle`; deposito real al empezar; regla `plan_aisle` con `stockout` y `aisle`; 20 pasadas de intercambio; recorrido en S. Para `swap_search`: $\lambda = 0{,}5$,
vecindad `bay` y plan inicial `demand_greedy`.

| bloque | que barre | planes | depositos |
|---|---|---|---|
| A | slotting por ventana; con cada uno, regla y techo; hoy | todos | los cuatro (`big` con 5 planes y ventanas 1, 4, 16) |
| R | regla de reposicion | vigente, `demand_greedy`, `aisle_fill`, `merchant_aisle_fill` | los cuatro (`big` con 2 planes) |
| F | frecuencia | `demand_greedy`, `aisle_fill`, `merchant_aisle_fill` | `small`, `medium`, `medium-alpha-1.3` |
| Q | parametros de `swap_search` | `swap_search` | `small`, `medium`, `medium-alpha-1.3` |
| B | pasadas del re-batching | `demand_greedy`, `merchant_aisle_fill` | `small`, `medium`, `medium-alpha-1.3` |
| G | recorrido | `demand_greedy`, `aisle_fill`, `merchant_aisle_fill` | `small`, `medium`, `medium-alpha-1.3` |

`random` y `objective_greedy` no usan el plan, asi que se corren una sola vez por deposito, con el plan
vigente. El filtro de afinidad `none` no se incluye: en la grilla 05 tardo 7,7 horas por plan en
`medium`. `swap_search` no corre en `big` por tiempo.

In [2]:
DEPOSITOS = ["small", "medium", "medium-alpha-1.3", "big"]
SIN_BIG = ["small", "medium", "medium-alpha-1.3"]
VENTANAS = (1, 2, 4, 8, 16)

REGLA = lambda w, **kw: EvalSpec(window=w, **kw)                                   # configuracion base
TECHO = lambda w: EvalSpec(window=w, start="plan", replenishment="home")
HOY = lambda w, **kw: EvalSpec(window=w, replenishment="random", **kw)

VIGENTE = SlottingSpec("vigente")
RAPIDOS = [SlottingSpec("demand_greedy"), SlottingSpec("aisle_fill"), SlottingSpec("merchant_aisle_fill")]
GANADOR = SlottingSpec("merchant_aisle_fill")
BILEVEL = [SlottingSpec("bilevel", clustering=c, zone_cost=z) for c in ("merchant", "demand_class") for z in ("dock", "aisle_order")]
SWAP_BASE = SlottingSpec("swap_search", lam=0.5, neighborhood="bay", seed="demand_greedy")

**Bloque A.** Cada plan con la regla y con el techo en todas las ventanas; el vigente, ademas, con hoy.

In [3]:
tareas = []
for d in DEPOSITOS:
    ventanas = VENTANAS if d != "big" else (1, 4, 16)
    planes = [VIGENTE] + RAPIDOS + BILEVEL
    if d != "big":
        planes += [SlottingSpec("linear_assignment"), SWAP_BASE]
    else:
        planes = [VIGENTE] + RAPIDOS + [SlottingSpec("bilevel", clustering="merchant", zone_cost="aisle_order")]
    for p in planes:
        evals = [REGLA(w) for w in ventanas] + [TECHO(w) for w in ventanas]
        if p == VIGENTE:
            evals += [HOY(w) for w in ventanas]
        tareas.append(Task("A", InstanceSpec(d), p, tuple(evals)))

**Bloque R.** Las cuatro combinaciones de la regla (la base incluida, para tener todo en una tabla), con
ventanas 1 y 4; y, una vez por deposito, `objective_greedy` con $\lambda$ = 0, 0,5 y 1 y `random`.

In [4]:
VARIANTES_REGLA = [dict(rule_trigger=t, rule_target=g) for t in ("stockout", "always") for g in ("aisle", "location")]
for d in DEPOSITOS:
    planes = [VIGENTE] + RAPIDOS if d != "big" else [VIGENTE, GANADOR]
    for p in planes:
        evals = [REGLA(w, **v) for w in (1, 4) for v in VARIANTES_REGLA]
        if p == VIGENTE:
            evals += [EvalSpec(window=w, replenishment="objective_greedy", replenishment_lam=lam)
                      for w in (1, 4) for lam in (0.0, 0.5, 1.0)]
            evals += [HOY(w) for w in (1, 4)]
        tareas.append(Task("R", InstanceSpec(d), p, tuple(evals)))

**Bloque F.** Las cuatro definiciones de frecuencia con los tres planes que la usan para ordenar, ventanas 1 y 4.

In [5]:
for d in SIN_BIG:
    for f in ("pick_lines", "unique_orders", "unique_batches", "total_units"):
        tareas += [Task("F", InstanceSpec(d, demand_metric=f), p, (REGLA(1), REGLA(4))) for p in RAPIDOS]

**Bloque Q.** Cada fila cambia un solo parametro de `swap_search` respecto de la base, que esta en el
bloque A. Ventanas 1, 4 y 16, con la regla y con el techo.

In [6]:
# (factor, valor, cambios en la instancia, cambios en el plan)
VARIANTES_QAP = [
    ("lambda", 0.0, {}, {"lam": 0.0}), ("lambda", 0.25, {}, {"lam": 0.25}), ("lambda", 0.75, {}, {"lam": 0.75}),
    ("distancia D", "walking", {"pair_distance": "walking"}, {}),
    ("metrica de afinidad", "cosine", {"affinity_metric": "cosine"}, {}),
    ("metrica de afinidad", "cooccurrence", {"affinity_metric": "cooccurrence"}, {}),
    ("unidad de co-ocurrencia", "batch_id", {"cooccurrence_col": "batch_id"}, {}),
    ("filtro", "top_k:5", {"affinity_filter": "top_k:5"}, {}),
    ("filtro", "top_k:20", {"affinity_filter": "top_k:20"}, {}),
    ("filtro", "mutual_top_k:10", {"affinity_filter": "mutual_top_k:10"}, {}),
    ("normalizacion", False, {"normalize": False}, {}),
    ("vecindad", "neighbor", {}, {"neighborhood": "neighbor"}),
    ("plan inicial", "aisle_fill", {}, {"seed": "aisle_fill"}),
    ("plan inicial", "merchant_aisle_fill", {}, {"seed": "merchant_aisle_fill"}),
]
EVAL_Q = tuple([REGLA(w) for w in (1, 4, 16)] + [TECHO(w) for w in (1, 4, 16)])
variante_de = {}                                            # clave de la tarea -> (factor, valor)
for d in SIN_BIG:
    for factor, valor, en_instancia, en_plan in VARIANTES_QAP:
        t = Task("Q", replace(InstanceSpec(d), **en_instancia), replace(SWAP_BASE, **en_plan), EVAL_Q)
        tareas.append(t)
        variante_de[t.key] = (factor, valor)

**Bloques B y G.** Solo construir contra construir e intercambiar; y el recorrido `snake` (con su propio hoy).

In [7]:
for d in SIN_BIG:
    tareas += [Task("B", InstanceSpec(d), p, tuple(REGLA(w, batching_passes=n) for w in (2, 4, 8, 16) for n in (0, 20)))
               for p in (SlottingSpec("demand_greedy"), GANADOR)]
    tareas += [Task("G", InstanceSpec(d), p, tuple(REGLA(w, routing=r) for w in (1, 4) for r in ("s_shape", "snake")))
               for p in RAPIDOS]
    tareas.append(Task("G", InstanceSpec(d), VIGENTE, tuple(HOY(1, routing=r) for r in ("s_shape", "snake"))))

In [8]:
resumen = pd.DataFrame([{"bloque": t.block, "deposito": t.instance.dataset, "evaluaciones": len(t.evals)} for t in tareas])
print("tareas:", len(tareas), "| evaluaciones:", resumen["evaluaciones"].sum())
resumen.pivot_table(index="deposito", columns="bloque", values="evaluaciones", aggfunc="sum", fill_value=0)

tareas: 145 | evaluaciones: 906


bloque,A,B,F,G,Q,R
deposito,,,,,,
big,33,0,0,0,0,24
medium,105,16,24,14,84,40
medium-alpha-1.3,105,16,24,14,84,40
small,105,16,24,14,84,40


## 4. La corrida

`run_tasks` corre las tareas en paralelo, un deposito por vez, y guarda cada resultado en disco apenas
termina; si la corrida se corta, al volver a ejecutar retoma donde quedo. Primero corre todo menos
`swap_search` y despues `swap_search`, que tarda de 1 minuto a casi 5 horas por plan en `medium`, para
tener antes el resto de los resultados. Los procesos en paralelo estan limitados por la memoria: `big` ocupa unos 3
GB y corre de a uno. Las tareas con solver (`bilevel`) corren de a una por la licencia. El avance queda
en `progreso.log`.

In [9]:
def segundos_esperados(t):
    # Para ordenar: las tareas largas primero, asi los procesos terminan parejo.
    if t.slotting.method == "swap_search":
        return {"small": 300, "medium": 2400, "medium-alpha-1.3": 2400}.get(t.instance.dataset, 0)
    return 30 * len(t.evals)


TRABAJOS = {"small": 3, "medium": 2, "medium-alpha-1.3": 2, "big": 1}
sin_qap = [t for t in tareas if t.slotting.method != "swap_search"]
con_qap = [t for t in tareas if t.slotting.method == "swap_search"]
for grupo in (sin_qap, con_qap):
    estado = run_tasks(grupo, OUT, n_jobs=TRABAJOS, expected_seconds=segundos_esperados)
    print(estado["status"].value_counts().to_dict() if len(estado) else "todas las tareas ya tenian resultado")

todas las tareas ya tenian resultado
todas las tareas ya tenian resultado


**Registro de experimentos.** Cada configuracion (instancia, plan y evaluacion) queda en
`results/registro.db` con sus parametros, sus resultados (caminata promedio y final, contra hoy,
espera, mudanzas) y una descripcion; una configuracion que esta en varios bloques se registra una sola
vez. Las que ya estaban no se repiten. Se mira con `mlflow ui --backend-store-uri sqlite:///results/registro.db`.

In [10]:
BLOQUES = {"A": "slotting por ventana", "R": "regla de reposicion", "F": "frecuencia",
           "Q": "parametros del QAP", "B": "pasadas del re-batching", "G": "recorrido"}
print("configuraciones nuevas en el registro:", register_results(OUT, blocks=BLOQUES))

2026/10/29 12:42:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/29 12:42:20 INFO mlflow.store.db.utils: Updating database tables


configuraciones nuevas en el registro: 807


In [11]:
# Resultados: una fila por evaluacion, con el nombre del plan y la variante del bloque Q.
res = load_results(OUT)
res = res[res["task"].isin({t.key for t in tareas})].copy()


def nombre_plan(r):
    if r["slot_method"] == "bilevel":
        return f"bilevel {r['slot_clustering']}/{r['slot_zone_cost']}"
    return r["slot_method"]


def nombre_regla(r):
    if r["start"] == "plan":
        return "techo"
    if r["replenishment"] == "plan_aisle":
        return f"regla {r['rule_trigger']}/{r['rule_target']}"
    if r["replenishment"] == "objective_greedy":
        return f"objective_greedy lambda={r['replenishment_lam']}"
    return "hoy (random)"


res["plan"] = res.apply(nombre_plan, axis=1)
res["regla"] = res.apply(nombre_regla, axis=1)
res["factor"] = res["task"].map(lambda k: variante_de.get(k, ("", ""))[0])
res["valor"] = res["task"].map(lambda k: str(variante_de.get(k, ("", ""))[1]))

# Hoy de cada deposito y recorrido: vigente, reposicion random, FIFO.
hoy = (res.query("plan == 'vigente' and replenishment == 'random' and window == 1")
          .groupby(["dataset", "routing"])[["mean_distance", "last14_distance"]].first())
indice = pd.MultiIndex.from_frame(res[["dataset", "routing"]])
res["promedio vs hoy (%)"] = 100 * (res["mean_distance"].to_numpy() / hoy["mean_distance"].reindex(indice).to_numpy() - 1)
res["final vs hoy (%)"] = 100 * (res["last14_distance"].to_numpy() / hoy["last14_distance"].reindex(indice).to_numpy() - 1)
print("evaluaciones con resultado:", len(res), "de", sum(len(t.evals) for t in tareas))
hoy

evaluaciones con resultado: 906 de 906


mean_distance  last14_distance
dataset          routing                                
big              s_shape   49303.573481     49298.672377
medium           s_shape   17824.000000     17824.000000
                 snake     18303.138122     18298.476190
medium-alpha-1.3 s_shape   17824.000000     17824.000000
                 snake     18297.922652     18296.571429
small            s_shape    6400.000000      6400.000000
                 snake      6663.558621      6661.741176

## 5. Controles

- **Errores**: tareas que fallaron (el detalle queda en `progreso.log`).
- **Techo**: con el plan aplicado y `home` ningun SKU deja su pasillo, asi que la caminata simulada
  tiene que ser igual a la que calcula el re-batching.
- **Reproduccion**: `merchant_aisle_fill` con la regla base y ventana 4 en `medium` tiene que dar lo
  mismo que en `correct_base/01_transicion` (13.991,2 pulgadas por batch).
- **Faltantes**: lineas sin stock, por deposito.

In [12]:
log = (OUT / "progreso.log").read_text().splitlines() if (OUT / "progreso.log").exists() else []
print("tareas con error en el log:", sum(1 for l in log if " error " in l))
techo = res.query("start == 'plan' and routing == 's_shape'")
print("diferencia maxima en el techo, simulacion - algoritmo:", round((techo["mean_distance"] - techo["static_final"]).abs().max(), 3))
rep = res.query("block == 'A' and dataset == 'medium' and plan == 'merchant_aisle_fill' and regla == 'regla stockout/aisle' and window == 4")
print("reproduccion del 01 (13991.2):", rep["mean_distance"].round(1).tolist())
res.groupby("dataset")["shortfall_lines"].agg(["min", "max"]).rename(columns={"min": "faltantes min", "max": "faltantes max"})

tareas con error en el log: 0
diferencia maxima en el techo, simulacion - algoritmo: 0.0
reproduccion del 01 (13991.2): [13991.2]


,faltantes min,faltantes max
dataset,,
big,8,42
medium,0,0
medium-alpha-1.3,0,0
small,0,0


In [13]:
# Cuanto tarda cada evaluacion (segundos), por deposito y punto de partida.
res.pivot_table(index="dataset", columns=["start", "window"], values="sim_seconds", aggfunc="median").round(0)

start            plan                      real                         
window             1    2    4    8    16    1     2      4     8     16
dataset                                                                 
big               8.0  NaN  8.0  NaN  8.0  42.0   NaN  121.0   NaN  46.0
medium            4.0  4.0  4.0  4.0  4.0  22.0  83.0   48.0  31.0  16.0
medium-alpha-1.3  4.0  4.0  4.0  4.0  4.0  20.0  66.0   43.0  27.0  18.0
small             1.0  1.0  1.0  1.0  1.0   3.0   9.0    6.0   4.0   3.0

## 6. Bloque A: slotting por ventana

**Final** (ultimos 14 dias) contra hoy, con la regla base, por plan y ventana. Es la tabla principal.

In [14]:
A = res.query("block == 'A'")
A.query("regla == 'regla stockout/aisle'").pivot_table(index=["dataset", "plan"], columns="window",
                                                        values="final vs hoy (%)").round(1)

window                                              1     2     4     8     16
dataset          plan                                                         
big              aisle_fill                       -7.4   NaN -29.1   NaN -38.4
                 bilevel merchant/aisle_order     -5.7   NaN -36.5   NaN -53.0
                 demand_greedy                    -8.0   NaN -32.0   NaN -40.8
                 merchant_aisle_fill              -6.0   NaN -36.5   NaN -53.1
                 vigente                           0.0   NaN -16.6   NaN -31.8
medium           aisle_fill                       -1.0 -10.5 -16.8 -19.9 -22.6
                 bilevel demand_class/aisle_order -0.9 -11.3 -17.3 -20.5 -21.8
                 bilevel demand_class/dock        -0.6 -11.0 -17.4 -22.5 -24.8
                 bilevel merchant/aisle_order     -0.3 -13.2 -26.6 -33.2 -36.8
                 bilevel merchant/dock            -0.0  -8.7 -17.3 -24.2 -28.7
                 demand_greedy                    -0.6 -11.1 -17.1 -21.7 -24.0
                 linear_assignment                -0.7 -11.0 -16.8 -21.2 -24.4
                 merchant_aisle_fill              -0.4 -13.8 -27.4 -34.1 -37.1
                 swap_search                      -0.3 -12.5 -21.7 -27.1 -30.4
                 vigente                           0.0  -1.7 -11.0 -15.1 -18.0
medium-alpha-1.3 aisle_fill                       -1.7 -11.4 -17.8 -20.3 -22.1
                 bilevel demand_class/aisle_order -1.6 -11.8 -17.7 -21.3 -23.6
                 bilevel demand_class/dock        -1.4 -14.1 -19.5 -24.4 -27.3
                 bilevel merchant/aisle_order     -0.2 -14.3 -27.8 -34.6 -37.7
                 bilevel merchant/dock            -0.0  -9.2 -17.8 -24.8 -29.9
                 demand_greedy                    -1.4 -13.6 -20.5 -24.6 -27.3
                 linear_assignment                -1.3 -13.2 -19.7 -23.7 -26.3
                 merchant_aisle_fill              -0.4 -15.6 -28.9 -35.0 -38.6
                 swap_search                      -0.8 -14.6 -22.1 -27.9 -30.9
                 vigente                           0.0  -1.5 -10.6 -15.5 -17.8
small            aisle_fill                        0.0   0.0   0.0   0.0   0.0
                 bilevel demand_class/aisle_order  0.0   0.0   0.0   0.0   0.0
                 bilevel demand_class/dock         0.0   0.0   0.0   0.0   0.0
                 bilevel merchant/aisle_order      0.0  -0.2  -0.3  -0.3  -0.1
                 bilevel merchant/dock             0.0   0.0   0.0   0.0   0.0
                 demand_greedy                     0.0  -0.1  -0.1   0.0   0.0
                 linear_assignment                 0.0   0.0   0.0   0.0   0.0
                 merchant_aisle_fill               0.0  -0.1  -0.2  -0.1  -0.1
                 swap_search                       0.0   0.0   0.0   0.0   0.0
                 vigente                           0.0   0.0   0.0   0.0   0.0

El **promedio** de todo el test, que incluye la transicion:

In [15]:
A.query("regla == 'regla stockout/aisle'").pivot_table(index=["dataset", "plan"], columns="window",
                                                        values="promedio vs hoy (%)").round(1)

window                                              1     2     4     8     16
dataset          plan                                                         
big              aisle_fill                       -4.8   NaN -26.2   NaN -35.0
                 bilevel merchant/aisle_order     -3.1   NaN -30.7   NaN -46.2
                 demand_greedy                    -5.4   NaN -28.2   NaN -37.4
                 merchant_aisle_fill              -3.3   NaN -30.8   NaN -46.4
                 vigente                          -0.0   NaN -16.5   NaN -31.8
medium           aisle_fill                       -0.4  -8.2 -12.7 -15.0 -16.7
                 bilevel demand_class/aisle_order -0.4  -8.6 -13.0 -15.6 -16.8
                 bilevel demand_class/dock        -0.3  -8.9 -14.0 -17.2 -19.1
                 bilevel merchant/aisle_order     -0.1 -10.0 -20.8 -27.7 -30.8
                 bilevel merchant/dock            -0.0  -7.3 -14.6 -20.4 -23.8
                 demand_greedy                    -0.3  -9.1 -13.7 -17.0 -18.6
                 linear_assignment                -0.3  -8.9 -13.9 -16.8 -19.1
                 merchant_aisle_fill              -0.2 -10.4 -21.5 -28.3 -31.1
                 swap_search                      -0.2  -9.6 -16.6 -21.1 -23.7
                 vigente                           0.0  -1.6 -10.8 -15.2 -17.8
medium-alpha-1.3 aisle_fill                       -0.7  -9.2 -13.7 -16.1 -17.1
                 bilevel demand_class/aisle_order -0.6  -9.4 -13.9 -16.8 -18.1
                 bilevel demand_class/dock        -0.6 -10.8 -15.6 -19.3 -21.2
                 bilevel merchant/aisle_order     -0.1 -10.9 -22.3 -29.0 -32.2
                 bilevel merchant/dock            -0.0  -7.9 -15.3 -21.4 -25.0
                 demand_greedy                    -0.5 -10.6 -16.1 -19.6 -21.1
                 linear_assignment                -0.6 -10.5 -15.8 -19.2 -21.1
                 merchant_aisle_fill              -0.2 -11.5 -23.3 -29.6 -33.0
                 swap_search                      -0.3 -11.4 -17.6 -22.3 -25.3
                 vigente                           0.0  -1.5 -10.8 -15.4 -17.8
small            aisle_fill                       -0.0  -0.1  -0.1  -0.1  -0.1
                 bilevel demand_class/aisle_order -0.0  -0.1  -0.1  -0.0  -0.0
                 bilevel demand_class/dock         0.0  -0.0  -0.0  -0.0  -0.0
                 bilevel merchant/aisle_order      0.0  -0.1  -0.1  -0.1  -0.0
                 bilevel merchant/dock             0.0   0.0   0.0   0.0   0.0
                 demand_greedy                    -0.0  -0.0  -0.0  -0.0  -0.0
                 linear_assignment                 0.0  -0.0  -0.0  -0.0  -0.0
                 merchant_aisle_fill               0.0  -0.1  -0.1  -0.0  -0.0
                 swap_search                      -0.0  -0.0  -0.0  -0.0  -0.0
                 vigente                           0.0   0.0   0.0   0.0   0.0

El **techo** de cada plan, aplicado de una vez, y **hoy** con re-batching (sin plan):

In [16]:
pd.concat({(r, m): A[A["regla"] == r].pivot_table(index=["dataset", "plan"], columns="window", values=f"{m} vs hoy (%)")
           for r in ("techo", "hoy (random)") for m in ("final", "promedio")}).round(1)

window                                                                     1     2     4     8     16
                      dataset          plan                                                          
techo        final    big              aisle_fill                       -24.0   NaN -48.6   NaN -58.6
                                       bilevel merchant/aisle_order     -17.8   NaN -58.8   NaN -74.7
                                       demand_greedy                    -16.4   NaN -40.6   NaN -50.6
                                       merchant_aisle_fill              -18.9   NaN -59.8   NaN -75.8
                                       vigente                            0.0   NaN -16.6   NaN -31.8
                      medium           aisle_fill                        -6.4 -24.5 -32.8 -37.7 -40.2
                                       bilevel demand_class/aisle_order  -4.6 -23.1 -30.1 -34.9 -38.1
                                       bilevel demand_class/dock         -1.3 -14.9 -22.4 -28.7 -33.0
                                       bilevel merchant/aisle_order      -2.1 -29.5 -43.8 -56.1 -61.2
                                       bilevel merchant/dock             -0.1 -10.7 -22.1 -30.7 -35.8
                                       demand_greedy                     -1.3 -15.1 -22.4 -27.7 -32.0
                                       linear_assignment                 -1.7 -15.8 -22.2 -28.6 -32.4
                                       merchant_aisle_fill               -3.0 -32.9 -46.3 -57.9 -61.9
                                       swap_search                       -1.1 -23.9 -33.9 -41.9 -46.6
                                       vigente                            0.0  -1.7 -11.0 -15.1 -18.0
                      medium-alpha-1.3 aisle_fill                       -10.8 -28.3 -36.5 -40.0 -42.1
                                       bilevel demand_class/aisle_order  -8.8 -26.2 -33.2 -38.0 -40.7
                                       bilevel demand_class/dock         -3.6 -19.3 -26.9 -32.7 -36.3
                                       bilevel merchant/aisle_order      -1.5 -29.3 -44.0 -56.0 -60.6
                                       bilevel merchant/dock             -0.1 -11.4 -23.2 -31.0 -36.2
                                       demand_greedy                     -2.8 -19.3 -27.2 -33.0 -36.8
                                       linear_assignment                 -3.5 -19.2 -26.4 -32.1 -35.6
                                       merchant_aisle_fill               -2.8 -33.3 -48.1 -58.3 -62.0
                                       swap_search                       -2.3 -24.1 -35.2 -42.7 -48.1
                                       vigente                            0.0  -1.5 -10.6 -15.5 -17.8
                      small            aisle_fill                        -1.0  -5.8  -7.2  -9.0 -10.8
                                       bilevel demand_class/aisle_order  -0.6  -5.2  -7.1  -6.7  -7.9
                                       bilevel demand_class/dock         -0.3  -0.9  -0.9  -0.9  -0.9
                                       bilevel merchant/aisle_order       0.0 -13.1 -26.2 -30.4 -31.8
                                       bilevel merchant/dock              0.0   0.0  -0.1   0.0   0.0
                                       demand_greedy                     -0.1  -1.0  -1.1  -0.7  -0.7
                                       linear_assignment                 -0.4  -1.2  -1.1  -0.7  -0.7
                                       merchant_aisle_fill               -0.6  -3.9 -28.3 -35.3 -36.9
                                       swap_search                       -0.1  -4.6  -5.7  -7.1  -7.7
                                       vigente                            0.0   0.0   0.0   0.0   0.0
             promedio big              aisle_fill                       -24.1   NaN -48.6   NaN -58.8
                                       bilevel merchant/aisle_order     -17.7   NaN -59.1   NaN -74.7
                      

Cuanto del plan entro al final del test (demanda en su pasillo del plan) y pasillos por orden, con ventana 4:

In [17]:
A.query("regla == 'regla stockout/aisle' and window == 4").pivot_table(
    index="plan", columns="dataset", values=["final_in_plan", "final_aisles_per_order", "aisles_per_order"]).round(2)

aisles_per_order                               final_aisles_per_order                               final_in_plan                              
dataset                                       big medium medium-alpha-1.3 small                    big medium medium-alpha-1.3 small           big medium medium-alpha-1.3 small
plan                                                                                                                                                                            
aisle_fill                                   2.15   1.94             1.76  1.45                   2.62   2.34             2.46  1.85          0.44   0.64             0.61  0.74
bilevel demand_class/aisle_order              NaN   2.20             2.07  1.54                    NaN   2.36             2.52  1.85           NaN   0.72             0.70  0.76
bilevel demand_class/dock                     NaN   2.39             2.27  1.69                    NaN   2.32             2.36  1.92           NaN   0.81             0.83  0.80
bilevel merchant/aisle_order                 1.06   1.07             1.07  1.08                   1.49   1.40             1.30  1.53          0.84   0.88             0.89  0.86
bilevel merchant/dock                         NaN   1.79             1.74  1.69                    NaN   1.78             1.82  1.89           NaN   0.88             0.88  0.85
demand_greedy                                2.70   2.40             2.27  1.70                   2.74   2.36             2.36  1.80          0.63   0.81             0.81  0.84
linear_assignment                             NaN   2.39             2.27  1.70                    NaN   2.38             2.34  1.85           NaN   0.81             0.80  0.82
merchant_aisle_fill                          1.04   1.03             1.01  1.36                   1.53   1.34             1.20  1.61          0.84   0.88             0.89  0.86
swap_search                                   NaN   1.53             1.54  1.63                    NaN   1.78             1.86  1.89           NaN   0.83             0.83  0.84
vigente                                      3.22   2.94             2.94  2.45                   3.10   2.94             2.90  2.48          1.00   1.00             1.00  1.00

**Con FIFO casi ningun plan gana.** En `medium` y `medium-alpha-1.3` ninguno pasa de 1,7%, porque un
batch FIFO junta 25 ordenes de casi todos los pasillos y entra a los 8 igual. En `big`, con 16
pasillos, los planes por demanda llegan a 8,0%.

**Con re-batching ganan los dos planes que agrupan por merchant,** en todos los depositos y desde la
ventana 2. Con ventana 4, `merchant_aisle_fill` gana 27,4% al final en `medium` y
`bilevel merchant/aisle_order` 26,6%; en `medium-alpha-1.3`, 28,9% y 27,8%; en `big`, 36,5% los dos.
Los planes por demanda (`demand_greedy`, `linear_assignment`, `aisle_fill`) y los otros tres binivel
quedan entre 16,8% y 17,4% en `medium`, y `swap_search` en el medio (21,7%). El vigente con la regla ya
gana 11,0%: dejar de mudar SKUs al azar vale algo por si solo, y es mas que re-armar los batches sin
plan (hoy con re-batching, 4,1% en promedio).

**Partiendo del deposito real, el plan llega a algo mas de la mitad del techo.** Con ventana 4, el
plan por merchant gana 27,4% al final contra 46,3% de su techo en `medium`, y 36,5% contra 59,8% en
`big`. El plan entra solo por las mudanzas de las reposiciones, y los SKUs que casi no se agotan no se
mudan: al final del test el 88% de la demanda esta en su pasillo del plan (84% en `big`) y una orden
toca 1,34 pasillos, contra 1,03 con el plan completo. El promedio, que incluye la transicion, es 21,5%.

**En `small` ningun plan se mueve** (a lo sumo 0,3%), aunque el techo del plan por merchant con ventana
4 es 28,3% al final: con 4 pasillos, mientras quede demanda fuera de su pasillo casi cualquier batch entra a
los 4.

## 7. Bloque R: regla de reposicion

Final contra hoy y mudanzas por las reposiciones, para las cuatro variantes de la regla, con ventanas 1
y 4. Abajo, las politicas que no usan plan.

In [18]:
R = res.query("block == 'R'")
reglas = R[R["replenishment"] == "plan_aisle"]
pd.concat({"final vs hoy (%)": reglas.pivot_table(index=["dataset", "plan", "window"], columns="regla", values="final vs hoy (%)"),
           "promedio vs hoy (%)": reglas.pivot_table(index=["dataset", "plan", "window"], columns="regla", values="promedio vs hoy (%)"),
           "mudanzas": reglas.pivot_table(index=["dataset", "plan", "window"], columns="regla", values="relocations")}, axis=1).round(1)

final vs hoy (%)                                                                    promedio vs hoy (%)                                                                     \
regla                                       regla always/aisle regla always/location regla stockout/aisle regla stockout/location  regla always/aisle regla always/location regla stockout/aisle regla stockout/location   
dataset          plan                window                                                                                                                                                                                
big              merchant_aisle_fill 1                    -8.1                  -0.0                 -6.0                    -5.9                -5.6                  -0.6                 -3.3                    -3.3   
                                     4                   -37.3                 -13.3                -36.5                   -36.5               -32.5                 -16.7                -30.8                   -30.8   
                 vigente             1                     0.0                   0.0                  0.0                     0.0                -0.0                  -0.0                 -0.0                    -0.0   
                                     4                   -16.6                 -16.6                -16.6                   -16.6               -16.5                 -16.5                -16.5                   -16.5   
medium           aisle_fill          1                    -0.2                   0.0                 -1.0                    -1.0                -0.1                  -0.0                 -0.4                    -0.4   
                                     4                   -12.4                  -2.5                -16.8                   -12.5                -7.7                  -1.7                -12.7                    -9.9   
                 demand_greedy       1                    -0.5                   0.0                 -0.6                    -0.4                -0.3                  -0.0                 -0.3                    -0.2   
                                     4                   -14.1                  -2.2                -17.1                   -11.1                -9.9                  -1.5                -13.7                    -8.9   
                 merchant_aisle_fill 1                    -0.5                   0.0                 -0.4                    -0.4                -0.3                  -0.0                 -0.2                    -0.2   
                                     4                   -27.1                  -3.5                -27.4                   -27.1               -21.0                  -4.0                -21.5                   -21.3   
                 vigente             1                     0.0                   0.0                  0.0                     0.0                 0.0                   0.0                  0.0                     0.0   
                                     4                   -11.0                 -11.0                -11.0                   -11.0               -10.8                 -10.8                -10.8                   -10.8   
medium-alpha-1.3 aisle_fill          1                    -0.2                  -0.0                 -1.7                    -1.4                -0.1                  -0.0                 -0.7                    -0.5   
                                     4                   -12.6                  -1.9                -17.8                   -11.9                -8.7                  -1.6                -13.7                   -10.2   
                 demand_greedy       1                    -1.3                  -0.0                 -1.4                    -0.7                -0.7                  -0.0                 -0.5                    -0.2   
                                     4                   -17.7                  -1.8  

In [19]:
R[R["replenishment"] != "plan_aisle"].pivot_table(index=["dataset", "regla"], columns="window",
                                                  values=["final vs hoy (%)", "relocations"]).round(1)

final vs hoy (%)       relocations         
window                                                      1     4           1        4
dataset          regla                                                                  
big              hoy (random)                             0.0 -13.4     29860.0  28125.0
                 objective_greedy lambda=0.0             -4.4 -28.7     19833.0  19833.0
                 objective_greedy lambda=0.5             -7.2 -29.9     13756.0  13684.0
                 objective_greedy lambda=1.0             -1.9 -21.6      5630.0   5593.0
medium           hoy (random)                             0.0  -4.2     13608.0  12570.0
                 objective_greedy lambda=0.0             -0.1 -20.2      8668.0   8654.0
                 objective_greedy lambda=0.5             -0.4 -15.6      5639.0   5545.0
                 objective_greedy lambda=1.0             -0.0 -11.7      1771.0   1809.0
medium-alpha-1.3 hoy (random)                             0.0  -4.1     13437.0  12216.0
                 objective_greedy lambda=0.0             -0.1 -21.8      8511.0   8404.0
                 objective_greedy lambda=0.5             -0.8 -19.6      5161.0   5153.0
                 objective_greedy lambda=1.0             -0.0 -12.4      1716.0   1735.0
small            hoy (random)                             0.0   0.0      1357.0   1329.0
                 objective_greedy lambda=0.0              0.0  -0.1      1367.0   1382.0
                 objective_greedy lambda=0.5              0.0   0.0       808.0    806.0
                 objective_greedy lambda=1.0              0.0   0.0       368.0    368.0

**La regla base, mover solo al agotarse y a un hueco de su pasillo, es la mejor o empata con todos
los planes.** Con ventana 4 en `medium`:
- mover tambien cuando al SKU le quedan unidades (`always/aisle`) empata con el plan por merchant
  (27,1% contra 27,4%) y pierde 3 a 4,5 puntos con los planes por demanda, con 1,5 a 1,8 veces mas
  mudanzas;
- apuntar a la ubicacion exacta del plan (`stockout/location`) empata con el plan por merchant y pierde
  4 a 6 puntos con los otros, con 2,2 a 3,6 veces mas mudanzas;
- las dos cosas juntas (`always/location`) casi borran la ganancia (3,5% con el plan por merchant) y
  hacen 26.424 mudanzas, el doble que hoy.

En `big`, `always/aisle` gana 0,8 puntos mas que la base (37,3% contra 36,5%) con 58% mas mudanzas.

**Sin plan, la mejor reposicion es `objective_greedy` con la afinidad.** Con ventana 4 y $\lambda = 0$,
que decide solo por afinidad, gana 20,2% en `medium` y 21,8% en `medium-alpha-1.3`; con $\lambda = 1$,
solo por demanda, 11,7% y 12,4%. En `big`, 28,7% con $\lambda = 0$ y 29,9% con 0,5. La afinidad de los
datos, que es el merchant, sirve aun sin plan, y hace menos mudanzas que `random` (8.654 contra 12.570 en
`medium`, con la misma ventana), pero queda por debajo del plan por merchant con la regla.

## 8. Bloque F: frecuencia

Final contra hoy segun la definicion de $f$, con la regla base.

In [20]:
F = res.query("block == 'F'")
F.pivot_table(index=["dataset", "plan", "window"], columns="demand_metric", values="final vs hoy (%)").round(1)

demand_metric                                pick_lines  total_units  unique_batches  unique_orders
dataset          plan                window                                                        
medium           aisle_fill          1             -1.0         -1.0            -1.0           -1.0
                                     4            -16.8        -16.5           -16.5          -16.8
                 demand_greedy       1             -0.6         -0.5            -0.4           -0.6
                                     4            -17.1        -16.8           -16.6          -17.1
                 merchant_aisle_fill 1             -0.4         -0.4            -0.4           -0.4
                                     4            -27.4        -27.6           -27.7          -27.4
medium-alpha-1.3 aisle_fill          1             -1.7         -1.6            -1.7           -1.7
                                     4            -17.8        -17.9           -17.9          -17.8
                 demand_greedy       1             -1.4         -1.3            -1.4           -1.4
                                     4            -20.5        -19.7           -20.3          -20.5
                 merchant_aisle_fill 1             -0.4         -0.4            -0.4           -0.4
                                     4            -28.9        -28.7           -28.5          -28.9
small            aisle_fill          1              0.0          0.0             0.0            0.0
                                     4              0.0          0.0             0.0            0.0
                 demand_greedy       1              0.0          0.0             0.0            0.0
                                     4             -0.1          0.0            -0.1           -0.1
                 merchant_aisle_fill 1              0.0          0.0             0.0            0.0
                                     4             -0.2         -0.2            -0.2           -0.2

**La frecuencia da igual.** Las cuatro definiciones quedan a menos de un punto entre si con los tres
planes, todos los depositos y las dos ventanas; la mayor diferencia es 0,8 puntos (`demand_greedy` en
`medium-alpha-1.3` con ventana 4). `pick_lines` y `unique_orders` son la misma cuenta, porque ninguna
orden repite un SKU. Se queda `pick_lines`.

## 9. Bloque Q: parametros del QAP

Cada fila cambia un parametro de `swap_search` respecto de la base (primera fila de cada deposito).
Final contra hoy con la regla, y techo, con ventanas 1, 4 y 16.

In [21]:
base_q = res.query("block == 'A' and slot_method == 'swap_search' and window in [1, 4, 16]").assign(factor="base", valor="")
Q = pd.concat([base_q, res.query("block == 'Q'")])
Q["medida"] = np.where(Q["regla"] == "techo", "techo (promedio)", "regla (final)")
Q["vs hoy (%)"] = np.where(Q["regla"] == "techo", Q["promedio vs hoy (%)"], Q["final vs hoy (%)"])
orden = ["base"] + list(dict.fromkeys(f for f, *_ in VARIANTES_QAP))
tabla_q = Q.pivot_table(index=["dataset", "factor", "valor"], columns=["medida", "window"], values="vs hoy (%)").round(1)
tabla_q.reindex(sorted(tabla_q.index, key=lambda i: (i[0], orden.index(i[1]) if i[1] in orden else 99, i[2])))

medida                                                       regla (final)             techo (promedio)            
window                                                                  1     4     16               1     4     16
dataset          factor                  valor                                                                     
medium           base                                                 -0.3 -21.7 -30.4             -1.1 -34.0 -46.7
                 lambda                  0.0                          -0.4 -22.2 -31.6             -1.3 -35.2 -50.6
                                         0.25                         -0.3 -22.3 -31.0             -1.1 -36.1 -49.5
                                         0.75                         -0.2 -22.1 -30.0             -1.5 -36.3 -46.8
                 distancia D             walking                      -0.4 -19.7 -28.7             -1.4 -28.7 -38.5
                 metrica de afinidad     cooccurrence                 -0.6 -19.7 -25.7             -2.6 -34.6 -41.1
                                         cosine                       -0.2 -24.0 -34.2             -1.1 -38.0 -54.2
                 unidad de co-ocurrencia batch_id                     -0.3 -17.1 -23.6             -1.1 -25.8 -35.3
                 filtro                  mutual_top_k:10              -0.3 -21.1 -30.4             -1.3 -34.0 -44.1
                                         top_k:20                     -0.2 -24.2 -34.5             -0.9 -35.1 -50.8
                                         top_k:5                      -0.3 -19.7 -27.8             -1.4 -33.1 -44.0
                 normalizacion           False                        -0.6 -17.8 -25.1             -1.7 -26.4 -36.0
                 vecindad                neighbor                     -0.7 -17.6 -23.7             -0.8 -25.1 -34.6
                 plan inicial            aisle_fill                   -0.7 -21.5 -29.6             -3.6 -41.1 -55.6
                                         merchant_aisle_fill          -0.3 -25.8 -36.6             -1.2 -44.1 -61.7
medium-alpha-1.3 base                                                 -0.8 -22.1 -30.9             -2.4 -35.0 -47.8
                 lambda                  0.0                          -0.1 -23.7 -33.0             -1.4 -38.8 -53.6
                                         0.25                         -0.4 -22.6 -32.1             -1.8 -35.2 -49.2
                                         0.75                         -0.5 -21.9 -31.1             -2.6 -34.0 -44.3
                 distancia D             walking                      -0.8 -20.8 -29.8             -2.4 -29.6 -39.5
                 metrica de afinidad     cooccurrence                 -1.4 -19.2 -25.4             -5.8 -35.4 -41.9
                                         cosine                       -0.3 -24.6 -33.8             -1.6 -38.2 -52.2
                 unidad de co-ocurrencia batch_id                     -1.2 -18.9 -24.8             -3.7 -30.1 -38.0
                 filtro                  mutual_top_k:10              -0.4 -20.9 -29.9             -2.1 -31.9 -41.3
                                         top_k:20                     -0.3 -24.8 -35.2             -1.1 -36.8 -52.2
                                         top_k:5                      -0.8 -21.5 -29.8             -2.6 -33.2 -42.1
                 normalizacion           False                        -1.2 -20.6 -27.7             -3.3 -28.7 -37.6
                 vecindad                neighbor                     -0.9 -19.2 -25.4             -2.1 -29.4 -38.0
                 plan inicial            aisle_fill                   -1.0 -22.4 -31.4             -5.6 -40.0 -51.0
                                         merchant_aisle_fill          -0.2 -28.3 -37.9             -1.8 -45.6 -62.0
small            base                                                  0.0   0.0   0.0             -0.1  -4.4  -7.9
                 lambda                  0.0            

In [22]:
# Cuanto tardo cada plan de swap_search (minutos), por deposito.
Q.groupby(["dataset", "factor", "valor"])["plan_seconds"].first().div(60).round(1).unstack("dataset")

dataset                                      medium  medium-alpha-1.3  small
factor                  valor                                               
base                                          279.5              49.9   18.0
distancia D             walking                24.4              42.6    9.4
filtro                  mutual_top_k:10         7.9              13.3    1.3
                        top_k:20              103.3             281.4   53.9
                        top_k:5                10.2              20.1    2.3
lambda                  0.0                   266.4              33.5   10.0
                        0.25                   34.0              49.1   11.3
                        0.75                   41.1              47.0   15.4
metrica de afinidad     cooccurrence          223.0             710.8   67.7
                        cosine                 50.7              71.3   17.1
normalizacion           False                  35.3              72.7   22.7
plan inicial            aisle_fill             44.1              42.8   25.4
                        merchant_aisle_fill    17.3              38.6    4.4
unidad de co-ocurrencia batch_id               69.1              57.0    8.9
vecindad                neighbor                0.5               0.6    0.4

**Ningun `swap_search` le gana al plan por merchant.** El mejor arranca del plan por merchant y gana
25,8% con ventana 4 en `medium` y 28,3% en `medium-alpha-1.3`, contra 27,4% y 28,9% del plan por
merchant solo. Los intercambios que bajan el objetivo del QAP sacan algunas ordenes de su pasillo
(1,08 pasillos por orden contra 1,03), y lo mismo pasa con el techo: 44,1% contra 47,1% en `medium`.

**Arrancando de `demand_greedy`, lo que mas suma es lo que acerca el plan al de merchant.** La base
gana 21,7% en `medium`, con 1,53 pasillos por orden. Con 20 vecinos por SKU gana 24,2% y con `cosine`
24,0%, porque esos planes dejan las ordenes en 1,21 a 1,27 pasillos. $\lambda$ entre 0 y 0,75 da casi
lo mismo (21,7% a 22,3%).

**Empeoran:** la vecindad `neighbor` (17,6%), la co-ocurrencia por batch (17,1%), no normalizar
(17,8%) y, 2 puntos debajo de la base, la distancia `walking`, `cooccurrence` y 5 vecinos (19,7%
cada uno). `medium-alpha-1.3` da un orden parecido, y en `small` ninguna variante gana mas de 0,5%
partiendo del deposito real.

**Es caro:** cada plan tarda de 1 minuto a casi 5 horas en `medium`, y con `cooccurrence` casi 12
horas en `medium-alpha-1.3`, contra segundos del plan por merchant.

## 10. Bloque B: algoritmo de batching

Final contra hoy solo construyendo (0 pasadas) y construyendo e intercambiando (20).

In [23]:
B = res.query("block == 'B'")
B.pivot_table(index=["dataset", "plan", "window"], columns="batching_passes", values="final vs hoy (%)").round(1)

batching_passes                                0     20
dataset          plan                window            
medium           demand_greedy       2      -10.3 -11.1
                                     4      -15.9 -17.1
                                     8      -20.6 -21.7
                                     16     -23.4 -24.0
                 merchant_aisle_fill 2      -12.6 -13.8
                                     4      -26.7 -27.4
                                     8      -33.3 -34.1
                                     16     -36.6 -37.1
medium-alpha-1.3 demand_greedy       2      -12.4 -13.6
                                     4      -18.3 -20.5
                                     8      -22.7 -24.6
                                     16     -25.0 -27.3
                 merchant_aisle_fill 2      -13.6 -15.6
                                     4      -27.5 -28.9
                                     8      -34.3 -35.0
                                     16     -37.9 -38.6
small            demand_greedy       2       -0.1  -0.1
                                     4       -0.1  -0.1
                                     8        0.0   0.0
                                     16       0.0   0.0
                 merchant_aisle_fill 2       -0.1  -0.1
                                     4       -0.2  -0.2
                                     8       -0.1  -0.1
                                     16      -0.1  -0.1

**Intercambiar ordenes despues de construir los batches suma poco, pero siempre:** entre 0,5 y 2,3
puntos en `medium` y `medium-alpha-1.3` (0,7 con el plan por merchant y ventana 4 en `medium`). Se
quedan las 20 pasadas.

## 11. Bloque G: recorrido

Final contra hoy con cada recorrido. Cada recorrido se compara con su propio hoy (vigente, `random`,
FIFO, medido con ese recorrido). El re-batching agrupa siempre con la caminata en S.

In [24]:
G = res.query("block == 'G' and plan != 'vigente'")
G.pivot_table(index=["dataset", "plan", "window"], columns="routing", values="final vs hoy (%)").round(1)

routing                                      s_shape  snake
dataset          plan                window                
medium           aisle_fill          1          -1.0   -9.6
                                     4         -16.8  -19.0
                 demand_greedy       1          -0.6  -10.1
                                     4         -17.1  -19.8
                 merchant_aisle_fill 1          -0.4   -6.0
                                     4         -27.4  -23.2
medium-alpha-1.3 aisle_fill          1          -1.7  -10.9
                                     4         -17.8  -20.5
                 demand_greedy       1          -1.4  -12.4
                                     4         -20.5  -23.5
                 merchant_aisle_fill 1          -0.4   -7.4
                                     4         -28.9  -25.3
small            aisle_fill          1           0.0   -1.1
                                     4           0.0   -1.0
                 demand_greedy       1           0.0   -1.4
                                     4          -0.1   -1.5
                 merchant_aisle_fill 1           0.0   -0.4
                                     4          -0.2   -0.6

**Con `snake` la posicion dentro del pasillo empieza a contar.** Contra su propio hoy, con FIFO los
planes por demanda ganan 9,6% y 10,1% en `medium`, porque `snake` cobra cuanto se entra en cada pasillo
y esos planes ponen los SKUs mas pedidos al frente; el plan por merchant gana 6,0%. Con ventana 4 el
plan por merchant sigue primero, con menos ventaja: 23,2% contra 19,0% a 19,8% (en S, 27,4% contra
16,8% a 17,1%). `medium-alpha-1.3` da lo mismo. El recorrido de la operacion es S, y con `snake` el
orden de los planes no cambia.

## 12. Que queda y que se descarta

Las cifras son el final contra hoy con ventana 4 en `medium`, salvo que se diga otra cosa.

| factor | queda | se descarta | por que |
|---|---|---|---|
| slotting | `merchant_aisle_fill` | planes por demanda, binivel por clases, `swap_search` | 27,4% contra 16,8% a 21,7% (6 y 9) |
| | `bilevel merchant/aisle_order`, como alternativa | | 26,6%: la misma idea con un solver |
| regla | `stockout/aisle` | `location`, `always/location` | la mejor o empata, con menos mudanzas (7) |
| ventana | 4 como referencia | FIFO | con FIFO ningun plan gana mas de 1% en `medium`; la ventana la fija la espera que se tolere |
| frecuencia | `pick_lines` | las otras tres | menos de un punto de diferencia (8) |
| afinidad del QAP | si se usa: `cosine` o 20 vecinos, desde el plan por merchant | `neighbor`, `batch_id`, sin normalizar | aun asi no supera al plan por merchant (9) |
| batching | 20 pasadas | 0 | 0,5 a 2,3 puntos (10) |
| recorrido | `s_shape` | | `snake` no cambia el orden de los planes (11) |
| reposicion sin plan | `objective_greedy` con $\lambda = 0$ | `random` | 20,2% contra 4,2%, pero debajo del plan por merchant (7) |

**`small` no sirve para elegir:** con 4 pasillos ningun plan gana mas de 0,3%, y con ventana 4 una
orden espera 843 minutos.

La configuracion que queda (el plan por merchant, la regla base, ventana 4 y 20 pasadas) es la de
`04_experimento_ganador`.